# 03. CNN desde cero: NumPy (CPU) contra CuPy (GPU) con kernels CUDA propios

Proyecto 3 CC3069. Las tres CNN del proyecto escritas **sin autograd ni framework**: forward y backward
a mano, Adam a mano. El mismo algoritmo en tres variantes, para ver qué aporta cada nivel de CUDA:

| Variante | Dónde corre | Código |
|---|---|---|
| NumPy | CPU, 1 hilo de Python (BLAS multihilo en la multiplicación) | `src/scratch/numpy_cnn.py` |
| CuPy, solo operaciones de CuPy (`ASL_CUPY_KERNELS=0`) | GPU: cuBLAS + kernels elementales de CuPy | `src/scratch/cupy_cnn.py` |
| CuPy con **kernels CUDA propios** (`ASL_CUPY_KERNELS=1`) | GPU: cuBLAS + `cp.RawKernel` escritos en CUDA C | `src/scratch/cupy_cnn.py` |

La convolución es `im2col` + multiplicación de matrices (cuBLAS). Los kernels propios reemplazan lo que
rodea a esa multiplicación: `im2col`, `col2im`, ReLU, MaxPool 2x2 y Adam.

Secciones:

0. Configuración y verificación del hardware
1. Fase A: preparación de datos en CPU (caché de uint8)
2. Los kernels CUDA: mapeo de hilos y bloques
3. Verificación contra PyTorch y contra la versión de CuPy sin kernels
4. Microbenchmark por kernel
5. Benchmark de entrenamiento (NumPy, CuPy, CuPy con kernels)
6. Tablas de tiempos y speedup

**Cómo correrlo:** ejecutar las celdas en orden. Las banderas de la sección 0 permiten saltarse partes
largas. Mismos parámetros que el resto del proyecto: lote 64, float32, semilla 42, Adam lr = 1e-3,
5 épocas fijas, sin early stopping, 3 corridas, época 1 descartada (calentamiento).

## 0. Configuración y verificación del hardware

In [1]:
# NOTA: corrida de evidencia reducida (ver banderas). Los tiempos del informe salen de results/*.json
import os, sys, json, time, glob, subprocess, inspect
from pathlib import Path

import numpy as np
import pandas as pd

# Raíz del repo (el notebook vive en notebooks/)
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "results"

# ---------------- Parámetros del experimento (igual que el resto del proyecto) ----------------
RUNS = 1                    # corridas independientes por configuración
EPOCHS = 5                  # épocas fijas, sin early stopping
WORKERS = os.cpu_count()    # procesos para decodificar JPEG (fase de datos)

# ---------------- Banderas: poner False para saltarse una sección ----------------
RUN_TESTS = True            # sección 3, pytest tests/test_scratch.py (~15 s)
RUN_MICRO = True            # sección 4, microbenchmark por kernel (~10 s)
RUN_SCRATCH_NUMPY = False    # sección 5, NumPy (lento: ~7 min por corrida, solo CNN-1)
RUN_CUPY_OPS = True         # sección 5, CuPy sin kernels propios (~2 min por corrida)
RUN_CUPY_KERNELS = True     # sección 5, CuPy con kernels propios (~1.5 min por corrida)

def sh(cmd, env=None, cwd=ROOT):
    # Ejecuta un comando y muestra su salida en la celda, línea a línea
    e = {**os.environ, **(env or {})}
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=e, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))} (código {p.returncode})")

print(f"Raíz del proyecto: {ROOT}")
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | hilos lógicos de CPU: {WORKERS}")

Raíz del proyecto: /home/Tono/Downloads/asl-cuda
Python 3.13.9 | NumPy 2.2.6 | hilos lógicos de CPU: 20


In [2]:
# Verificación de que el entorno reconoce la GPU
import cupy as cp
props = cp.cuda.runtime.getDeviceProperties(0)
GPU_NAME = props["name"].decode()
print(f"CuPy {cp.__version__} | CUDA runtime {cp.cuda.runtime.runtimeGetVersion()}")
print(f"GPU: {GPU_NAME} | VRAM {props['totalGlobalMem'] / 2**30:.1f} GiB | "
      f"capacidad de cómputo {props['major']}.{props['minor']} | multiprocesadores {props['multiProcessorCount']}")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

CuPy 14.2.0 | CUDA runtime 12090
GPU: NVIDIA GeForce RTX 4070 SUPER | VRAM 11.6 GiB | capacidad de cómputo 8.9 | multiprocesadores 56
name, driver_version, memory.total [MiB]
NVIDIA GeForce RTX 4070 SUPER, 580.95.05, 12282 MiB



## 1. Fase A: preparación de datos (CPU)

Igual que en el resto del proyecto, **todo lo que toca archivos ocurre en la CPU**: leer el JPEG,
decodificar, redimensionar a 64x64. `cache_data.py` lo hace una sola vez con N procesos y guarda uint8
en `data/cache/`. La GPU solo ve tensores ya armados: el lote se arma en CPU (gather) y se copia a la GPU
dentro del ciclo de entrenamiento (H2D), sin subir todo el dataset de golpe.

In [3]:
sh([sys.executable, "src/scratch/cache_data.py", "--workers", WORKERS])

[data] Train 11,600 | Val 1,450 | Test 1,450


train: (11600, 3, 64, 64) en 0.6 s


val: (1450, 3, 64, 64) en 0.2 s
test: (1450, 3, 64, 64) en 0.2 s


## 2. Los kernels CUDA: mapeo de hilos y bloques

Todos usan el mismo esquema: grid 1D de bloques de 256 hilos, **un hilo por elemento de la salida**,
`idx = blockIdx.x * blockDim.x + threadIdx.x`, y `grid = ceil(elementos / 256)`. Se compilan con NVRTC la
primera vez que se piden (`cp.RawModule`).

| Kernel | Un hilo calcula | Nota de diseño |
|---|---|---|
| `im2col` | un elemento de `cols` (C·K·K, N·H·W) | lee con relleno de ceros (padding `same`) |
| `col2im` | un elemento de `dx` (N, C, H, W) | es un *gather*: suma las K·K contribuciones que le tocan, **sin atómicas** |
| `relu_fwd` / `relu_bwd` | un elemento | guarda la máscara como uint8 |
| `maxpool_fwd` / `maxpool_bwd` | un pixel de salida / de entrada | guarda cuál de las 4 posiciones ganó (empate: la primera, como PyTorch) |
| `adam_step` | un parámetro | actualiza m, v y p en **una sola pasada** por memoria |

La multiplicación de matrices sigue en cuBLAS (`cp.matmul`): no se reescribe, no se le gana a cuBLAS.

In [4]:
os.environ["ASL_CUPY_KERNELS"] = "1"
from scratch import cupy_cnn as cc

print(f"BLOCK = {cc.BLOCK} hilos por bloque | USE_KERNELS = {cc.USE_KERNELS}\n")
print(inspect.getsource(cc._launch))
print(cc._SRC[:cc._SRC.index("extern \"C\" __global__ void relu_fwd")])    # im2col y col2im

BLOCK = 256 hilos por bloque | USE_KERNELS = True

def _launch(name, total, *args):
    """Grid 1D: ceil(total / BLOCK) bloques de BLOCK hilos, un hilo por elemento."""
    grid = (int((total + BLOCK - 1) // BLOCK),)
    _k(name)(grid, (BLOCK,), args)


// im2col: un hilo por elemento de cols (C*K*K, N*H*W). Lee con relleno de ceros (padding 'same').
extern "C" __global__ void im2col(const float* x, float* cols,
                                  int N, int C, int H, int W, int K, long long total) {
    long long idx = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= total) return;
    long long HW = (long long)H * W, NHW = (long long)N * HW;
    long long col = idx % NHW, row = idx / NHW;          // fila: (c, i, j); columna: (n, y, x)
    int n = (int)(col / HW), r = (int)(col % HW);
    int y = r / W, xx = r % W;
    int KK = K * K, c = (int)(row / KK), ij = (int)(row % KK);
    int sy = y + ij / K - K / 2, sx = xx + ij % K - K / 2;
    float v = 0.0f;
    if (sy >= 0

## 3. Verificación

Un kernel que da resultados distintos no sirve, así que se verifica antes de medir:

- NumPy y CuPy contra PyTorch: logits, pérdida y gradientes de todas las capas.
- Kernels propios contra la versión de CuPy sin kernels (`cp` puro): `im2col`, `col2im`, ReLU, MaxPool y
  20 pasos de Adam, además de que ambas versiones aprenden igual.

In [5]:
if RUN_TESTS:
    sh([sys.executable, "-m", "pytest", "tests/test_scratch.py", "-v", "--tb=short", "-p", "no:cacheprovider"])

============================= test session starts ==============================
platform linux -- Python 3.13.9, pytest-9.2.0, pluggy-1.7.0 -- /usr/bin/python3
rootdir: /home/Tono/Downloads/asl-cuda
plugins: anyio-4.13.0


collecting ... collected 21 items

tests/test_scratch.py::test_params[CNN-1 Baseline-cnn1-cnn1] PASSED      [  4%]
tests/test_scratch.py::test_params[CNN-2 Profunda-cnn2-cnn2] PASSED      [  9%]
tests/test_scratch.py::test_params[CNN-3 Regularizada-cnn3-cnn3] PASSED  [ 14%]
tests/test_scratch.py::test_vs_torch[CNN-1 Baseline-cnn1-cnn1] PASSED    [ 19%]


tests/test_scratch.py::test_vs_torch[CNN-2 Profunda-cnn2-cnn2] PASSED    [ 23%]
tests/test_scratch.py::test_vs_torch[CNN-3 Regularizada-cnn3-cnn3] PASSED [ 28%]


tests/test_scratch.py::test_finite_differences PASSED                    [ 33%]
tests/test_scratch.py::test_cupy_available PASSED                        [ 38%]


tests/test_scratch.py::test_cupy_params PASSED                           [ 42%]


tests/test_scratch.py::test_cupy_vs_numpy_adam_20_steps PASSED           [ 47%]
tests/test_scratch.py::test_cupy_vs_torch[CNN-1 Baseline-cnn1-cnn1] PASSED [ 52%]


tests/test_scratch.py::test_cupy_vs_torch[CNN-2 Profunda-cnn2-cnn2] PASSED [ 57%]
tests/test_scratch.py::test_cupy_vs_torch[CNN-3 Regularizada-cnn3-cnn3] PASSED [ 61%]


tests/test_scratch.py::test_cupy_learns PASSED                           [ 66%]


tests/test_scratch.py::test_learns PASSED                                [ 71%]
tests/test_scratch.py::test_kernel_conv_vs_cupy_ops[3-8-3-16] PASSED     [ 76%]
tests/test_scratch.py::test_kernel_conv_vs_cupy_ops[4-6-5-12] PASSED     [ 80%]
tests/test_scratch.py::test_kernel_conv_vs_cupy_ops[2-5-3-7] PASSED      [ 85%]


tests/test_scratch.py::test_kernel_relu_maxpool_vs_cupy_ops[<lambda>0] PASSED [ 90%]
tests/test_scratch.py::test_kernel_relu_maxpool_vs_cupy_ops[<lambda>1] PASSED [ 95%]


tests/test_scratch.py::test_kernel_adam_vs_cupy_ops PASSED               [100%]

=============================== warnings summary ===============================
../../.local/lib/python3.13/site-packages/anyio/pytest_plugin.py:14
  /home/Tono/.local/lib/python3.13/site-packages/anyio/pytest_plugin.py:14: PytestRemovedIn10Warning: _pytest.python.CallSpec2 has been renamed to CallSpec.
  The CallSpec2 alias will be removed in pytest 10.
  Update imports to use CallSpec instead.
  See https://docs.pytest.org/en/stable/deprecations.html#callspec2-renamed
    from _pytest.python import CallSpec2

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
================== 21 passed, 1 warning in 303.53s (0:05:03) ===================


## 4. Microbenchmark por kernel

Cada operación con los kernels propios contra la misma operación escrita con CuPy, con tensores del tamaño
que ve CNN-2 en su primera capa convolucional (lote 64, 32 canales, 64x64). Se mide con eventos CUDA, 20
repeticiones tras el calentamiento. Esto responde si cada kernel *se paga solo* o si basta con CuPy.

In [6]:
if RUN_MICRO:
    def t_gpu(fn, reps=20):
        for _ in range(3):
            fn()                                # calentamiento (y compilación NVRTC)
        a, b = cp.cuda.Event(), cp.cuda.Event()
        a.record()
        for _ in range(reps):
            fn()
        b.record(); b.synchronize()
        return cp.cuda.get_elapsed_time(a, b) / reps          # ms

    def con_kernels(flag, fn):
        def run():
            cc.USE_KERNELS = flag
            return fn()
        return run

    rng = np.random.default_rng(0)
    N, CIN, COUT, H = 64, 32, 32, 64
    x = cp.asarray(rng.standard_normal((N, CIN, H, H), dtype=np.float32))
    dout = cp.asarray(rng.standard_normal((N, COUT, H, H), dtype=np.float32))
    dpool = cp.asarray(rng.standard_normal((N, COUT, H // 2, H // 2), dtype=np.float32))

    conv = cc.Conv2D(CIN, COUT, 3, np.random.default_rng(1))
    relu, pool = cc.ReLU(), cc.MaxPool2()
    adam = cc.Adam([(cp.zeros(1_000_000, cp.float32), cp.ones(1_000_000, cp.float32))])

    def conv_fb():                              # im2col + GEMM + col2im
        conv.forward(x, True)
        return conv.backward(dout)

    casos = {
        "Conv forward + backward (im2col, col2im)": conv_fb,
        "ReLU forward": lambda: relu.forward(x, True),
        "ReLU forward + backward": lambda: (relu.forward(x, True), relu.backward(dout)),
        "MaxPool 2x2 forward": lambda: pool.forward(x, True),
        "MaxPool 2x2 forward + backward": lambda: (pool.forward(x, True), pool.backward(dpool)),
        "Adam, 1 M parámetros": adam.step,
    }
    filas = []
    for nombre, fn in casos.items():
        t_ops = t_gpu(con_kernels(False, fn))
        t_krn = t_gpu(con_kernels(True, fn))
        filas.append({"Operación": nombre, "CuPy (ms)": round(t_ops, 3),
                      "Kernel propio (ms)": round(t_krn, 3), "Speedup": f"{t_ops / t_krn:.2f}x"})
    cc.USE_KERNELS = True
    micro = pd.DataFrame(filas)
    micro.to_csv(RESULTS / "scratch_microbench.csv", index=False)
    display(micro)

,Operación,CuPy (ms),Kernel propio (ms),Speedup
0,"Conv forward + backward (im2col, col2im)",6.519,4.517,1.44x
1,ReLU forward,0.264,0.178,1.48x
2,ReLU forward + backward,0.444,0.355,1.25x
3,MaxPool 2x2 forward,0.455,0.018,25.96x
4,MaxPool 2x2 forward + backward,0.902,0.205,4.40x
5,"Adam, 1 M parámetros",0.138,0.010,13.25x


## 5. Benchmark de entrenamiento

Mismo protocolo que `scripts/benchmark.py`, y mismo formato de salida para que `scripts/aggregate.py` lo tome:

| Archivo en `results/` | Configuración |
|---|---|
| `scratch_numpy_run<N>.json` | NumPy (solo CNN-1: CNN-2 y CNN-3 son ~7x más cómputo y tardarían más de una hora por corrida) |
| `scratch_cupy_run<N>.json` | CuPy sin kernels propios (`ASL_CUPY_KERNELS=0`) |
| `scratch_cupy_kernels_run<N>.json` | CuPy con kernels propios (`ASL_CUPY_KERNELS=1`) |

Cada corrida guarda los tiempos de gather (CPU), H2D (PCIe) y cómputo (forward + backward + Adam) por época.

In [7]:
if RUN_SCRATCH_NUMPY:
    for run in range(1, RUNS + 1):
        sh([sys.executable, "scripts/bench_scratch.py", "--backend", "numpy", "--run", run,
            "--epochs", EPOCHS, "--models", "CNN-1 Baseline"])

if RUN_CUPY_OPS:
    for run in range(1, RUNS + 1):
        sh([sys.executable, "scripts/bench_scratch.py", "--backend", "cupy", "--run", run,
            "--epochs", EPOCHS], env={"ASL_CUPY_KERNELS": "0"})

if RUN_CUPY_KERNELS:
    for run in range(1, RUNS + 1):
        sh([sys.executable, "scripts/bench_scratch.py", "--backend", "cupy", "--run", run,
            "--epochs", EPOCHS, "--tag", "_kernels"], env={"ASL_CUPY_KERNELS": "1"})

backend cupy, train 11600, test 1450

=== CNN-1 Baseline, 4,221,405 parámetros ===
  época 1: 1.33 s (gather 0.03, H2D 0.06, red 1.24), loss 3.0278
  época 2: 1.06 s (gather 0.03, H2D 0.06, red 0.97), loss 1.9471
  época 3: 1.06 s (gather 0.03, H2D 0.06, red 0.97), loss 1.3659
  época 4: 1.06 s (gather 0.03, H2D 0.06, red 0.97), loss 1.0403
  época 5: 1.06 s (gather 0.03, H2D 0.06, red 0.97), loss 0.8520
CNN-1 Baseline: 5.6 s, accuracy 84.07 %

=== CNN-2 Profunda, 519,069 parámetros ===
  época 1: 4.53 s (gather 0.03, H2D 0.06, red 4.44), loss 2.5259
  época 2: 4.53 s (gather 0.03, H2D 0.06, red 4.44), loss 1.3260
  época 3: 4.53 s (gather 0.03, H2D 0.06, red 4.44), loss 0.8452
  época 4: 4.53 s (gather 0.03, H2D 0.06, red 4.44), loss 0.6284
  época 5: 4.53 s (gather 0.03, H2D 0.06, red 4.44), loss 0.4390
CNN-2 Profunda: 22.7 s, accuracy 66.14 %

=== CNN-3 Regularizada, 448,925 parámetros ===
  época 1: 4.76 s (gather 0.03, H2D 0.06, red 4.67), loss 2.9471
  época 2: 4.67 s (gather 0.0

backend cupy, train 11600, test 1450

=== CNN-1 Baseline, 4,221,405 parámetros ===
  época 1: 0.77 s (gather 0.03, H2D 0.06, red 0.68), loss 3.0479
  época 2: 0.66 s (gather 0.03, H2D 0.06, red 0.57), loss 2.0176
  época 3: 0.66 s (gather 0.03, H2D 0.06, red 0.57), loss 1.4619
  época 4: 0.66 s (gather 0.03, H2D 0.06, red 0.57), loss 1.1375
  época 5: 0.66 s (gather 0.03, H2D 0.06, red 0.57), loss 0.9267
CNN-1 Baseline: 3.4 s, accuracy 81.38 %

=== CNN-2 Profunda, 519,069 parámetros ===
  época 1: 3.90 s (gather 0.03, H2D 0.06, red 3.81), loss 2.5217
  época 2: 3.90 s (gather 0.03, H2D 0.06, red 3.81), loss 1.3408
  época 3: 3.89 s (gather 0.03, H2D 0.06, red 3.80), loss 0.8638
  época 4: 3.89 s (gather 0.03, H2D 0.06, red 3.80), loss 0.6379
  época 5: 3.89 s (gather 0.03, H2D 0.06, red 3.80), loss 0.4580
CNN-2 Profunda: 19.5 s, accuracy 22.48 %

=== CNN-3 Regularizada, 448,925 parámetros ===
  época 1: 4.05 s (gather 0.03, H2D 0.06, red 3.96), loss 2.9477
  época 2: 4.05 s (gather 0.0

## 6. Tablas de tiempos y speedup

Speedup = `T_secuencial / T_paralelo`. Segundos por época sin la época 1, media y desviación entre corridas.

In [8]:
def cargar(patron):
    return [json.load(open(f)) for f in sorted(glob.glob(str(RESULTS / patron)))]

def por_epoca(runs, modelo, clave="epoch_seconds"):
    # media por corrida (sin la época 1) -> media y desviación entre corridas
    v = [np.mean(r["modelos"][modelo][clave][1:]) for r in runs if clave in r["modelos"].get(modelo, {})]
    return (np.mean(v), np.std(v, ddof=1) if len(v) > 1 else 0.0) if v else (np.nan, np.nan)

cfgs = {"NumPy CPU": cargar("scratch_numpy_run*.json"),
        "CuPy GPU (sin kernels propios)": cargar("scratch_cupy_run*.json"),
        "CuPy GPU (kernels propios)": cargar("scratch_cupy_kernels_run*.json")}
modelos = ["CNN-1 Baseline", "CNN-2 Profunda", "CNN-3 Regularizada"]

filas = []
for nombre, runs in cfgs.items():
    fila = {"Configuración": nombre, "Corridas": len(runs)}
    for m in modelos:
        mu, sd = por_epoca(runs, m)
        fila[m] = f"{mu:.3f} ± {sd:.3f}" if not np.isnan(mu) else "-"
    filas.append(fila)
print("Tiempo por época (s)")
display(pd.DataFrame(filas))

def speedup(num, den):
    out = {}
    for m in modelos:
        a, b = por_epoca(cfgs[num], m)[0], por_epoca(cfgs[den], m)[0]
        out[m] = f"{a / b:.2f}x" if not (np.isnan(a) or np.isnan(b)) else "-"
    return out

print("\nSpeedup por época")
display(pd.DataFrame({
    "CuPy (sin kernels) vs NumPy": speedup("NumPy CPU", "CuPy GPU (sin kernels propios)"),
    "CuPy (kernels) vs NumPy": speedup("NumPy CPU", "CuPy GPU (kernels propios)"),
    "CuPy (kernels) vs CuPy (sin kernels)": speedup("CuPy GPU (sin kernels propios)", "CuPy GPU (kernels propios)"),
}))

# Dónde se va el tiempo de una época con los kernels propios: gather (CPU), H2D (PCIe), cómputo (GPU)
filas = []
for m in modelos:
    runs = cfgs["CuPy GPU (kernels propios)"]
    g, h, s, t = (por_epoca(runs, m, k)[0] for k in
                  ("gather_seconds", "h2d_seconds", "step_seconds", "epoch_seconds"))
    filas.append({"Modelo": m, "gather CPU (s)": round(g, 3), "H2D (s)": round(h, 3),
                  "cómputo GPU (s)": round(s, 3), "época (s)": round(t, 3)})
print("\nFases de una época, CuPy con kernels propios")
display(pd.DataFrame(filas))

# La exactitud debe ser la misma con y sin kernels (mismos pesos iniciales, misma semilla)
acc = {n: {m: np.mean([r["modelos"][m]["accuracy"] for r in runs if m in r["modelos"]])
           for m in modelos if any(m in r["modelos"] for r in runs)} for n, runs in cfgs.items()}
print("\nExactitud en test")
display(pd.DataFrame(acc).T.map(lambda v: f"{v * 100:.2f} %" if v == v else "-"))

Tiempo por época (s)


,Configuración,Corridas,CNN-1 Baseline,CNN-2 Profunda,CNN-3 Regularizada
0,NumPy CPU,3,64.123 ± 5.070,-,-
1,CuPy GPU (sin kernels propios),3,1.084 ± 0.029,4.731 ± 0.243,4.846 ± 0.147
2,CuPy GPU (kernels propios),3,0.683 ± 0.026,4.082 ± 0.180,4.277 ± 0.212



Speedup por época


,CuPy (sin kernels) vs NumPy,CuPy (kernels) vs NumPy,CuPy (kernels) vs CuPy (sin kernels)
CNN-1 Baseline,59.14x,93.83x,1.59x
CNN-2 Profunda,-,-,1.16x
CNN-3 Regularizada,-,-,1.13x



Fases de una época, CuPy con kernels propios


,Modelo,gather CPU (s),H2D (s),cómputo GPU (s),época (s)
0,CNN-1 Baseline,0.029,0.067,0.586,0.683
1,CNN-2 Profunda,0.031,0.068,3.982,4.082
2,CNN-3 Regularizada,0.032,0.071,4.173,4.277



Exactitud en test


,CNN-1 Baseline,CNN-2 Profunda,CNN-3 Regularizada
NumPy CPU,83.59 %,-,-
CuPy GPU (sin kernels propios),84.07 %,66.14 %,40.76 %
CuPy GPU (kernels propios),81.38 %,22.48 %,36.48 %
